In [ ]:
import os
# # 70B 모델 로드하는 경우에만 이 셀 실행

# import os

# # 중요: transformers나 torch 같은 라이브러리를 import하기 전에 먼저 실행해야 합니다!
# os.environ['HF_HOME'] = '/mnt/c/hf_cache'

# 라이브러리 로드

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
from typing import Optional
import pandas as pd
from tqdm import tqdm
import re
import os

# 토큰 및 환경변수 설정
login(token=os.environ["HF_TOKEN"])

load_dotenv(override=True)  # .env 파일 우선 사용
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# 타임아웃 설정을 위한 환경 변수
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"  # 5분으로 타임아웃 증가

# 인코딩 문제 해결을 위한 환경 변수 설정
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"  # 불필요한 경고 메시지 억제

# 모델 로드

## medicine-chat-7B
- 8비트 양자화 모델 로드

In [ ]:
# 전역 변수로 모델 저장
loaded_model = None
loaded_tokenizer = None
device = None

def load_model_medichat():
    """모델을 한 번만 로드하는 함수"""
    global loaded_model, loaded_tokenizer, device
    
    if loaded_model is not None:
        return  # 이미 로드됨

    # 양자화 설정
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False
    )

    # 모델과 토크나이저 로드
    loaded_model = AutoModelForCausalLM.from_pretrained(
        "AdaptLLM/medicine-chat",
        quantization_config=quantization_config,
        device_map="auto",
        dtype=torch.float16,
        low_cpu_mem_usage=True
    )
    loaded_tokenizer = AutoTokenizer.from_pretrained("AdaptLLM/medicine-chat")
    device = next(loaded_model.parameters()).device
    
    print("모델 로딩 완료!")

load_model_medichat()

## med42-70B
- 4비트 양자화 모델 로드

In [ ]:
# 전역 변수로 모델 저장
loaded_model = None
loaded_tokenizer = None
device = None

def load_model_med42():
    """모델을 한 번만 로드하는 함수 (Llama 3 및 4-bit 양자화에 최적화)"""
    global loaded_model, loaded_tokenizer, device
    
    if loaded_model is not None:
        return  # 이미 로드됨

    # 4-bit 양자화 설정
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 # 연산 시 bfloat16 사용
    )

    # 모델과 토크나이저 로드
    loaded_model = AutoModelForCausalLM.from_pretrained(
        "m42-health/Llama3-Med42-70B",
        quantization_config=quantization_config,
        device_map="auto",
        dtype=torch.bfloat16, # 로딩 시 데이터 타입도 bfloat16으로 지정
        low_cpu_mem_usage=True
    )
    loaded_tokenizer = AutoTokenizer.from_pretrained("m42-health/Llama3-Med42-70B")
    device = loaded_model.device
    
    print("모델 로딩 완료!")

load_model_med42()

# 공통 함수

In [ ]:
def parse_final_answer(row, question_type, prompt_type):
    """행 데이터에서 최종 답변을 파싱하는 함수"""
    
    # 행에서 필요한 데이터 추출
    if prompt_type == 'cove':
        response = row.get('final_response', '')  # 또는 적절한 컬럼명
    elif prompt_type == 'ao':
        response = row.get('ans_raw', '')
    elif prompt_type == 'cot':
        response = row.get('cot', '')
    
    # 만약 row에 ques가 있다면 그것을 사용하고, 없다면 question을 사용한다.
    if question_type == 'RQ':
        if 'ques' in row:
            ques = row.get('ques', '')
        elif 'question' in row:
            ques = row.get('question', '')
        else:
            ques = None
            print(f"질문이 없습니다. 질문 컬럼명은 ques 또는 question이어야 합니다.")

    
    system = '''당신은 주어진 문제 풀이 과정 텍스트에서 최종 답만을 찾아야 합니다.

응답 형식은 다음을 준수하세요.
- 최종 답이 명확하게 제시되지 않거나 '없음'이라면 'none'으로 출력하세요.
'''
    if question_type == 'TFQ' or question_type == 'TFQ2':
        system += "- 최종 답 이외에 다른 설명이나 문장은 출력하지 않습니다.\n- 최종 답이 True이면 T, False이면 F로 출력하세요.\n- T, F, none 외에 다른 응답은 허용하지 않습니다."
    elif question_type == 'TFQ3':
        system += "- 최종 답 이외에 다른 설명이나 문장은 출력하지 않습니다.\n- 최종 답에서 처방과 근거가 적절하다고 판단했다면 T, 처방과 근거가 부적절하다고 판단했다면 F로 출력하세요.\n- T, F, none 외에 다른 응답은 허용하지 않습니다."
    elif question_type == 'MCQ':
        system += "- 최종 답 이외에 다른 설명이나 문장은 출력하지 않습니다.\n- 최종 답에 해당하는 선지를 대문자로 공백 없이 출력하세요."
    elif question_type == 'MAQ':
        system += "- 최종 답 이외에 다른 설명이나 문장은  출력하지 않습니다.\n- 최종 답에 해당하는 선지를 대문자로 공백 없이 출력하세요. 정답이 여러개일 경우 알파벳 순서로 쉼표로 연결하되, 공백없이 출력하세요."
    elif question_type == 'RQ':
        system += '''- 최종 답의 형태는 풀이자의 답(A/B/C/D), Alice의 답과의 일치 여부(T/F)를 공백 없이 쉼표로 연결해 출력해야 합니다.
    - 예를 들어 Alice의 답이 B이고, 풀이자의 답안에서도 B를 선택했다면 두 응답이 일치하므로 B,T로 공백 없이 출력합니다.
    - 예를 들어 Alice의 답이 C이고, 풀이자의 답안에서 A를 선택했다면 두 응답이 일치하지 않으므로 A,F로 공백 없이 출력합니다.
    - 예를 들어 Alice의 답이 A인데, 풀이자는 A,B,C,D 중 옳은 것이 없다고 판단했다면 none,F로 공백 없이 출력합니다.
    - 만약 Alice와 풀이자의 답안이 일치한다면, 문제 풀이 과정에서 실수로 "incorrect"라고 했더라도 T로 출력해야합니다.
- 출력 형식은 다음을 따르세요:
    - Alice's answer: (문제에 제시된 Alice의 답)
    - My answer: (풀이 과정에서 풀이자가 도출한 답)
    - Alice's answer is (same/not same) as my answer. Therefore, Alice's answer is (correct(T)/incorrect(F)).
    - <final_answer>(풀이자가 도출한 답),(T/F)</final_answer>
- 예시1:
    - Alice's answer: A
    - My answer: B
    - Alice's answer is not same as my answer. Therefore, Alice's answer is incorrect.
    - <final_answer>B,F</final_answer>
- 예시2:
    - Alice's answer: C
    - My answer: none of the above
    - Alice's answer is not same as my answer. Therefore, Alice's answer is incorrect.
    - <final_answer>none,F</final_answer>'''


    template = '''
문제에 대한 풀이 내용 중에서 최종적으로 도출한 답을 찾아 올바른 형식으로 출력하세요.

# 풀이:
{response}'''

    if question_type == "RQ":
        template = '''
문제에서 제시된 Alice의 답을 찾고, 풀이에서 풀이자의 답을 찾아 최종 답을 올바른 형식으로 출력하세요.
# 문제:
{ques}

# 풀이:
{response}     
'''

    custom_llm = ChatOpenAI(
        openai_api_key=OPENAI_API_KEY,
        model_name="gpt-4.1",
        max_tokens=2048
    )
    custom_prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("user", template)
    ])
    chain = custom_prompt | custom_llm
            
    # 입력 데이터 준비
    if question_type == "RQ":
        input_data = {
            'response': str(response),
            'ques': str(ques)
        }
    else:
        input_data = {
            'response': str(response)
        }
    # 체인 실행
    response = chain.invoke(input_data)
    return response.content.strip()

In [ ]:
def process_questions(data_path, output_path, prompt_type=None, model_name=None, question_type=None):
    """
    문제에 대한 응답을 생성하고 엑셀 파일로 저장하는 메인 함수
    - data_path : 문제 데이터 경로 (파일명 {question_type}.xlsx)
    - output_path : 출력 경로
    - prompt_type : cot, ao
    - model_name : medichat, med42
    - question_type : mcq, maq, rq, tfq, tfq2, tfq3
    """

    # 데이터 로드
    df = pd.read_excel(data_path + f'{question_type}.xlsx')
    print(f"데이터 로드 완료: {len(df)}개 행")
    

    if prompt_type == "cot":
        # 컬럼 초기화
        df['cot'] = ''
        df['ans'] = ''
    
        # 각 행에 대해 응답 생성
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="응답 생성 중"):
            try:
                # 응답 생성
                if model_name == 'medichat':
                    cot_response = generate_cot_medichat(row, question_type)
                elif model_name == 'med42':
                    cot_response = generate_cot_med42(row, question_type)                  
                df.at[idx, 'cot'] = cot_response

                # row에 cot 추가
                row['cot'] = cot_response

                # 정답 파싱
                parsed_answer = parse_final_answer(row, question_type, prompt_type)
                df.at[idx, 'ans'] = parsed_answer
            
                
            except Exception as e:
                print(f"\n오류 발생 (행 {idx+1}): {str(e)}")
                df.at[idx, 'cot'] = f"오류: {str(e)}"
                df.at[idx, 'ans'] = ""
    
    elif prompt_type == "ao":
        # 컬럼 초기화
        df['ans_raw'] = ''
        df['ans'] = ''
        
        # 각 행에 대해 응답 생성
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="응답 생성 중"):
            try:
                # 응답 생성
                if model_name == 'medichat':
                    ao_response = generate_ao_medichat(row, question_type)
                elif model_name == 'med42':
                    ao_response = generate_ao_med42(row, question_type)
                df.at[idx, 'ans_raw'] = ao_response

                # row에 ans_raw 추가
                row['ans_raw'] = ao_response

                # 정답 파싱
                cleaned_answer = parse_final_answer(row, question_type, prompt_type)
                df.at[idx, 'ans'] = cleaned_answer
            except Exception as e:
                print(f"\n오류 발생 (행 {idx+1}): {str(e)}")
                df.at[idx, 'ans_raw'] = ""
    
    # 결과 저장
    output_pre_path = output_path
    if not os.path.exists(output_pre_path):
        os.makedirs(output_pre_path)
    output_path = output_pre_path + f'{model_name}_{question_type}_{prompt_type}.xlsx'
    df.to_excel(output_path, index=False)
    print(f"\n결과 저장 완료: {output_path}")
    
    return df

# 프롬프트

AO

In [ ]:
# 시스템 프롬프트
ao_system_base = '''# Role
You are an expert in solving problems related to pharmaceuticals.

# Instructions
1. Solve the given problem and provide only the answer, not an explanation.
2. The two provided examples are for referencing the format and structure of the response. You must follow their style precisely.
3. Note: Do not solve the example problems again. You must only generate a response for the final, actual problem.
4. You must state the final answer in the following format.'''

ao_mcq_format = "\n  Write the answer without any explanation. (Example: A)"
ao_maq_format = "\n  Answer without space (Example: A,B,C)"
ao_rq_format = '''
        - if Alice's answer is same as your answer: "my answer is (your answer). Therefore, Alice's answer is correct."
        - if Alice's answer is not same as your answer: "my answer is (your answer). Therefore, Alice's answer is incorrect."'''
ao_tfq_format = "\n  Answer without any explanation. (Example: true/false)"
ao_tfq3_format = "\n  - Answer without any explanation. (Example: true/false)\n  - If both prescription and the provided rationale are clinically appropriate, answer true. Otherwise, answer false."

# 유저 프롬프트
ao_mcq_template_2shot = '''
### Example1:
Question: Which of the following best describes the classification of profenamine?\nOptions: A: Barbiturate\tB: Phenothiazine\tC: Benzothiazepine\tD: Benzodiazepine
Answer: B

### Example2:
Question: Which of the following is consistent with toxicity due to topiramate overdose?\nOptions: A: Hypotension, visual disturbances, and metabolic acidosis\tB: Leukopenia, rash, and parotitis\tC: Hypernatremia, alopecia, and epistaxis\tD: Diarrhea, cough, and fever
Answer: A

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

ao_maq_template_2shot = '''
### Example1:
Question: . Which of the following best describes the classification of profenamine? There may be one or more correct choices.\nOptions: A: Phenothiazine\nB: Phenothiazine\nC: Phenothiazines\nD: Benzodiazepine
Answer: A,B,C

### Example2:
Question: . Which of the following are consistent with toxicity due to topiramate overdose? There may be one or more correct choices.\nOptions: A: Hypotension, visual disturbances, and metabolic acidosis\nB: Leukopenia, rash, and parotitis\nC: Hypotension, visual disturbances, and metabolic acidosis\nD: Diarrhea, cough, and fever
Answer: A,C

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

ao_rq_template_2shot = '''
### Example1:
Question: Which of the following best describes the classification of profenamine?\nOptions: A: Barbiturate\tB: Phenothiazine\tC: Benzothiazepine\tD: Benzodiazepine. Alice's answer: B. Please determine whether her answer is correct, and if it's incorrect, provide the correct answer.
Answer: correct, the correct answer is B

### Example2:
Question: Which of the following is associated with the pharmacodynamics of moclobemide?\nOptions: A: Decreased serotonin levels\tB: Reversible inhibition of monoamine oxidase leading to increased monoamines\tC: Antihistaminic activity\tD: Inhibition of noradrenergic neurotransmission. Alice's answer: D. Please determine whether her answer is correct, and if it's incorrect, provide the correct answer.
Answer: incorrect, the correct answer is B

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

ao_tfq_template_2shot = '''
### Example1:
Question: . Statement: \"Benzothiazepine best describes the classification of profenamine.\", is the statement above true or false? Please answer true/false.
Answer: false

### Example2:
Question: . Statement: \"Inhibition of noradrenergic neurotransmission is not associated with the pharmacodynamics of moclobemide.\", is the statement above true or false? Please answer true/false.
Answer: true

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

ao_tfq2_template_2shot = '''
## Example1:
Case Scenario: A 45-year-old female with Major Depressive Disorder taking Fluoxetine presents with tremor, sweating, and agitation after starting a high-dose Tryptophan supplement.
Question: Statement: \"The patient's presentation is likely due to a pharmacodynamic interaction increasing the risk of serotonin syndrome.\", is the statement above true or false? Please answer true/false.
Answer: true

## Example2:
Case Scenario: A toddler ingested Tryptophan tablets and presents with diarrhea and hyperactive reflexes.
Question: Statement: \"These symptoms are inconsistent with Tryptophan overdose.\", is the statement above true or false? Please answer true/false.
Answer: false

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

ao_tfq3_template_2shot = '''
## Example1:
Case Scenario: A 35-year-old male with atypical depression has been managed on Phenelzine (an MAOI) for the past 6 months. He visits his psychiatrist reporting persistent low mood. The physician decides to prescribe L-Tryptophan 500 mg daily as an adjunct therapy to naturally boost serotonin levels.
Question: Is this prescription and the provided rationale clinically appropriate?
Answer: false

## Example2:
Case Scenario: A 28-year-old female presents with Seasonal Affective Disorder (SAD), describing symptoms that worsen every winter. She prefers nutritional precursors. Based on her diagnosis, the physician prescribes Tryptophan, explaining that it serves as a precursor for serotonin synthesis.
Question: Is this prescription and the provided rationale clinically appropriate?
Answer: true

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''



CoT

In [ ]:
# 시스템 프롬프트

cot_system_base = '''# Role
You are an expert in solving problems related to pharmaceuticals.

# Instructions
1.  Solve the given problem by thinking it through step-by-step. (Max 6 sentences)
2.  The two provided examples are for referencing the format and structure of the response. You must follow their style precisely.
3.  Note: Do not solve the example problems again. You must only generate a response for the final, actual problem.
4.  After your entire thinking process is described, you must state the final answer in the following format.'''

cot_mcq_format = "\n  Therefore, the answer is: (Answer)"
cot_maq_format = "\n  Therefore, the answer is: (Answer without space)"
cot_rq_format = '''
        - if Alice's answer is same as your answer: "my answer is (your answer). Therefore, Alice's answer is correct."
        - if Alice's answer is not same as your answer: "my answer is (your answer). Therefore, Alice's answer is incorrect."'''
cot_tfq_format = "\n  Therefore, the answer is: (true or false)"
cot_tfq3_format = "\n  Therefore, the answer is: (true or false)\n5. If both prescription and the provided rationale are clinically appropriate, answer true. Otherwise, answer false."



# 유저 프롬프트

cot_mcq_template_2shot = '''
### Example1:
Question: Which of the following best describes the classification of profenamine?\nOptions: A: Barbiturate\tB: Phenothiazine\tC: Benzothiazepine\tD: Benzodiazepine
Answer: Profenamine is a phenothiazine derivative used as an antiparkinsonian agent. Phenothiazines are a class of antipsychotic medications that share a common chemical structure characterized by a phenothiazine ring system. Option B (Phenothiazine) correctly describes the classification of profenamine. Option A (Barbiturate) is incorrect as barbiturates are sedative-hypnotic drugs with a different chemical structure. Option C (Benzothiazepine) is incorrect as this refers to a different class of drugs like diltiazem. Option D (Benzodiazepine) is incorrect as benzodiazepines are a separate class of drugs with sedative and anxiolytic properties. Therefore, the answer is: B"

### Example2:
Question: Which of the following is consistent with toxicity due to topiramate overdose?\nOptions: A: Hypotension, visual disturbances, and metabolic acidosis\tB: Leukopenia, rash, and parotitis\tC: Hypernatremia, alopecia, and epistaxis\tD: Diarrhea, cough, and fever
Answer: Topiramate is an antiepileptic drug that can cause various toxic effects in overdose. It can lead to metabolic acidosis due to its carbonic anhydrase inhibitory effects, which can cause bicarbonate loss and acid-base imbalance. Visual disturbances are common due to its effects on the central nervous system and potential impact on ocular function. Hypotension can occur due to its cardiovascular depressant effects. Option A correctly describes these characteristic toxic effects of topiramate overdose. Option B (Leukopenia, rash, and parotitis) is not typical of topiramate toxicity. Option C (Hypernatremia, alopecia, and epistaxis) is not associated with topiramate overdose. Option D (Diarrhea, cough, and fever) is not a typical manifestation of topiramate overdose toxicity. Therefore, the answer is: A

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

cot_maq_template_2shot = '''
### Example1:
Question: . Which of the following best describes the classification of profenamine? There may be one or more correct choices.\nOptions: A: Phenothiazine\nB: Phenothiazine\nC: Phenothiazines\nD: Benzodiazepine
Answer: Profenamine is a phenothiazine derivative used as an antiparkinsonian agent. Phenothiazines are a class of antipsychotic medications that share a common chemical structure. Options A, B, and C all refer to phenothiazine or phenothiazines, which correctly describe the classification of profenamine. Option D (Benzodiazepine) is incorrect as benzodiazepines are a different class of drugs with sedative and anxiolytic properties, not related to phenothiazines. Therefore, the answer is A,B,C

### Example2:
Question: . Which of the following are consistent with toxicity due to topiramate overdose? There may be one or more correct choices.\nOptions: A: Hypotension, visual disturbances, and metabolic acidosis\nB: Leukopenia, rash, and parotitis\nC: Hypotension, visual disturbances, and metabolic acidosis\nD: Diarrhea, cough, and fever
Answer: Topiramate is an antiepileptic drug that can cause various toxic effects in overdose. It can lead to metabolic acidosis due to its carbonic anhydrase inhibitory effects, which can cause bicarbonate loss and acid-base imbalance. Visual disturbances are common due to its effects on the central nervous system and potential impact on ocular function. Hypotension can occur due to its cardiovascular depressant effects. Options A and C both describe these characteristic toxic effects of topiramate overdose. Therefore, the answer is: A,C

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

cot_rq_template_2shot = '''
### Example1:
Question: Which of the following best describes the classification of profenamine?\nOptions: A: Barbiturate\tB: Phenothiazine\tC: Benzothiazepine\tD: Benzodiazepine. Alice's answer: B. Please determine whether her answer is correct, and if it's incorrect, provide the correct answer.
Answer: Profenamine is a phenothiazine derivative used as an antiparkinsonian agent. Phenothiazines are a class of antipsychotic medications that share a common chemical structure characterized by a phenothiazine ring system. Alice answered B (Phenothiazine), which is correct. Profenamine belongs to the phenothiazine class of drugs, not barbiturates (A), benzothiazepines (C), or benzodiazepines (D). Therefore, Alice's answer is correct. Therefore, the answer is: correct

### Example2:
Question: Which of the following is associated with the pharmacodynamics of moclobemide?\nOptions: A: Decreased serotonin levels\tB: Reversible inhibition of monoamine oxidase leading to increased monoamines\tC: Antihistaminic activity\tD: Inhibition of noradrenergic neurotransmission. Alice's answer: D. Please determine whether her answer is correct, and if it's incorrect, provide the correct answer.
Answer: Moclobemide is a reversible inhibitor of monoamine oxidase type A (RIMA). It works by inhibiting the enzyme monoamine oxidase, which normally breaks down monoamine neurotransmitters like serotonin, norepinephrine, and dopamine. By inhibiting this enzyme, moclobemide increases the levels of these neurotransmitters in the brain. Alice answered D (Inhibition of noradrenergic neurotransmission), which is incorrect. Moclobemide actually enhances noradrenergic neurotransmission by increasing norepinephrine levels, not inhibiting it. The correct answer is B (Reversible inhibition of monoamine oxidase leading to increased monoamines), which describes the actual mechanism of action of moclobemide. Therefore, the answer is: incorrect, the correct answer is B

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

cot_tfq_template_2shot = '''
### Example1:
Question: . Statement: \"Benzothiazepine best describes the classification of profenamine.\", is the statement above true or false? Please answer true/false.
Answer: Profenamine is a phenothiazine derivative used as an antiparkinsonian agent. Phenothiazines are a class of antipsychotic medications that share a common chemical structure characterized by a phenothiazine ring system. The statement claims that benzothiazepine best describes the classification of profenamine, which is false. Profenamine belongs to the phenothiazine class, not the benzothiazepine class. Benzothiazepines are a different class of drugs (like diltiazem) with different chemical structures and pharmacological properties. Therefore, the answer is: false

### Example2:
Question: . Statement: \"Inhibition of noradrenergic neurotransmission is not associated with the pharmacodynamics of moclobemide.\", is the statement above true or false? Please answer true/false.
Answer: Moclobemide is a reversible inhibitor of monoamine oxidase type A (RIMA). It works by inhibiting the enzyme monoamine oxidase, which normally breaks down monoamine neurotransmitters like serotonin, norepinephrine, and dopamine. By inhibiting this enzyme, moclobemide increases the levels of these neurotransmitters in the brain, including norepinephrine. The statement claims that inhibition of noradrenergic neurotransmission is not associated with moclobemide's pharmacodynamics, which is true. Moclobemide does not inhibit noradrenergic neurotransmission; instead, it enhances it by increasing norepinephrine levels. Therefore, the answer is: true

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

cot_tfq2_template_2shot = '''
## Example1:
Case Scenario: A 45-year-old female with Major Depressive Disorder taking Fluoxetine presents with tremor, sweating, and agitation after starting a high-dose Tryptophan supplement.
Question: Statement: "The patient's presentation is likely due to a pharmacodynamic interaction increasing the risk of serotonin syndrome.", is the statement above true or false? Please answer true/false.
Answer: Fluoxetine is a Selective Serotonin Reuptake Inhibitor (SSRI), and Tryptophan is a precursor to serotonin. Taking both concurrently can lead to excessive accumulation of serotonin in the body, a pharmacodynamic interaction known as Serotonin Syndrome. The patient's symptoms of tremor, sweating, and agitation are classic clinical signs of this condition. Therefore, the answer is: true.

## Example2:
Case Scenario: A toddler ingested Tryptophan tablets and presents with diarrhea and hyperactive reflexes.
Question: Statement: "These symptoms are inconsistent with Tryptophan overdose.", is the statement above true or false? Please answer true/false.
Answer: Clinical data on Tryptophan overdose indicates that symptoms can include agitation, gastrointestinal distress (such as diarrhea), and neuromuscular effects like overactive reflexes. Since the toddler is presenting with diarrhea and hyperactive reflexes, these signs are actually consistent with the known toxicity profile of Tryptophan. The statement claims they are "inconsistent," which is incorrect. Therefore, the answer is: false.

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''

cot_tfq3_template_2shot = '''
## Example1:
Case Scenario: A 35-year-old male with atypical depression has been managed on Phenelzine (an MAOI) for the past 6 months. He visits his psychiatrist reporting persistent low mood. The physician decides to prescribe L-Tryptophan 500 mg daily as an adjunct therapy to naturally boost serotonin levels.
Question: Is this prescription and the provided rationale clinically appropriate?
Answer: Phenelzine is a Monoamine Oxidase Inhibitor (MAOI), and L-Tryptophan is a precursor to serotonin. Administering Tryptophan to a patient taking an MAOI is contraindicated because it can lead to excessive accumulation of serotonin in the central nervous system. This specific combination significantly increases the risk of life-threatening Serotonin Syndrome. Therefore, the answer is: false.

## Example2:
Case Scenario: A 28-year-old female presents with Seasonal Affective Disorder (SAD), describing symptoms that worsen every winter. She prefers nutritional precursors. Based on her diagnosis, the physician prescribes Tryptophan, explaining that it serves as a precursor for serotonin synthesis.
Question: Is this prescription and the provided rationale clinically appropriate?
Answer: Tryptophan functions pharmacologically as a precursor for the synthesis of serotonin (5-HT). Clinical evidence supports its use as a treatment option for Seasonal Affective Disorder (winter-onset depression). Since the prescription aligns with the specific indication and the rationale regarding serotonin synthesis is scientifically accurate, the clinical decision is sound. Therefore, the answer is: true.

### Please complete the final sample with the same format as the given examples.
{ques}
Answer:'''



# Answer-only

In [ ]:
def generate_ao_medichat(row, question_type=None):
    """로드된 모델로 응답만 생성"""
    global loaded_model, loaded_tokenizer, device
    
    if loaded_model is None:
        raise ValueError("모델이 로드되지 않았습니다. 모델 로드 함수를 먼저 호출하세요.")

    # 프롬프트 정의
    system_base = ao_system_base
    MCQ_system = ao_system_base + ao_mcq_format
    MAQ_system = ao_system_base + ao_maq_format
    RQ_system = ao_system_base + ao_rq_format
    TFQ_system = ao_system_base + ao_tfq_format
    TFQ3_system = ao_system_base + ao_tfq3_format
    
    MCQ_template = ao_mcq_template_2shot
    MAQ_template = ao_maq_template_2shot   
    RQ_template = ao_rq_template_2shot
    TFQ_template = ao_tfq_template_2shot
    TFQ2_template = ao_tfq2_template_2shot
    TFQ3_template = ao_tfq3_template_2shot

    if question_type == "MCQ":
        system_prompt = MCQ_system
        user_template = MCQ_template
    elif question_type == "MAQ":
        system_prompt = MAQ_system
        user_template = MAQ_template
    elif question_type == "RQ":
        system_prompt = RQ_system
        user_template = RQ_template
    elif question_type == "TFQ":
        system_prompt = TFQ_system
        user_template = TFQ_template 
    elif question_type == "TFQ2":
        system_prompt = TFQ_system
        user_template = TFQ2_template
    elif question_type == "TFQ3":
        system_prompt = TFQ3_system
        user_template = TFQ3_template

    # 프롬프트 생성
    user_input = user_template.format(ques=str(row['ques']))
    prompt = f"<s>[INST] <<SYS>>{system_prompt}<</SYS>>\n\n{user_input} [/INST]"
    
    # 토크나이징 및 모델 추론
    inputs = loaded_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    outputs = loaded_model.generate(input_ids=inputs, max_length=2048)[0]

    answer_start = int(inputs.shape[-1])
    pred = loaded_tokenizer.decode(outputs[answer_start:], skip_special_tokens=True)

    return pred

In [ ]:
def generate_ao_med42(row, question_type=None):
    """로드된 모델로 응답만 생성"""
    global loaded_model, loaded_tokenizer, device
    
    if loaded_model is None:
        raise ValueError("모델이 로드되지 않았습니다. 모델 로드 함수를 먼저 호출하세요.")
    
    # 프롬프트 정의
    system_base = ao_system_base
    MCQ_system = ao_system_base + ao_mcq_format
    MAQ_system = ao_system_base + ao_maq_format
    RQ_system = ao_system_base + ao_rq_format
    TFQ_system = ao_system_base + ao_tfq_format
    TFQ3_system = ao_system_base + ao_tfq3_format
    
    MCQ_template = ao_mcq_template_2shot
    MAQ_template = ao_maq_template_2shot   
    RQ_template = ao_rq_template_2shot
    TFQ_template = ao_tfq_template_2shot
    TFQ2_template = ao_tfq2_template_2shot
    TFQ3_template = ao_tfq3_template_2shot

    if question_type == "MCQ":
        system_prompt = MCQ_system
        user_template = MCQ_template
    elif question_type == "MAQ":
        system_prompt = MAQ_system
        user_template = MAQ_template
    elif question_type == "RQ":
        system_prompt = RQ_system
        user_template = RQ_template
    elif question_type == "TFQ":
        system_prompt = TFQ_system
        user_template = TFQ_template 
    elif question_type == "TFQ2":
        system_prompt = TFQ_system
        user_template = TFQ2_template
    elif question_type == "TFQ3":
        system_prompt = TFQ3_system
        user_template = TFQ3_template

    # 프롬프트 생성
    user_input = user_template.format(ques=str(row['ques']))
    # Llama 3 형식의 대화 구조 생성
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]
    
    # Llama 3 전용 템플릿을 사용하여 프롬프트를 자동으로 생성
    # 이 방식은 모델이 기대하는 정확한 형식과 특수 토큰을 보장합니다.
    prompt = loaded_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 토크나이징 및 모델 추론
    inputs = loaded_tokenizer(prompt, return_tensors="pt").to(device)
    
    # 생성된 텍스트만 디코딩하기 위해 입력 토큰 길이를 저장
    input_length = inputs.input_ids.shape[1]
    
    outputs = loaded_model.generate(
        **inputs,
        max_new_tokens=2048, # max_length 대신 max_new_tokens 사용이 더 명확함
        eos_token_id=loaded_tokenizer.eos_token_id
    )
    
    # 새로 생성된 부분만 디코딩
    pred = loaded_tokenizer.decode(outputs[0, input_length:], skip_special_tokens=True)

    return pred

# CoT

In [ ]:
def generate_cot_medichat(row, question_type=None):
    """로드된 모델로 응답만 생성"""
    global loaded_model, loaded_tokenizer, device
    
    if loaded_model is None:
        raise ValueError("모델이 로드되지 않았습니다. 모델 로드 함수를 먼저 호출하세요.")
    
    system_base = cot_system_base
    MCQ_system = cot_system_base + cot_mcq_format
    MAQ_system = cot_system_base + cot_maq_format
    RQ_system = cot_system_base + cot_rq_format
    TFQ_system = cot_system_base + cot_tfq_format
    TFQ3_system = cot_system_base + cot_tfq3_format

    MCQ_template = cot_mcq_template_2shot
    MAQ_template = cot_maq_template_2shot
    RQ_template = cot_rq_template_2shot
    TFQ_template = cot_tfq_template_2shot    
    TFQ2_template = cot_tfq2_template_2shot
    TFQ3_template = cot_tfq3_template_2shot

    if question_type == "MCQ":
        system_prompt = MCQ_system
        user_template = MCQ_template
    elif question_type == "MAQ":
        system_prompt = MAQ_system
        user_template = MAQ_template
    elif question_type == "RQ":
        system_prompt = RQ_system
        user_template = RQ_template
    elif question_type == "TFQ":
        system_prompt = TFQ_system
        user_template = TFQ_template
    elif question_type == "TFQ2":
        system_prompt = TFQ_system
        user_template = TFQ2_template
    elif question_type == "TFQ3":
        system_prompt = TFQ3_system
        user_template = TFQ3_template

    # 프롬프트 생성
    user_input = user_template.format(ques=str(row['ques']))
    prompt = f"<s>[INST] <<SYS>>{system_prompt}<</SYS>>\n\n{user_input} [/INST]"
    
    # 토크나이징 및 모델 추론
    inputs = loaded_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    outputs = loaded_model.generate(input_ids=inputs, max_length=2048)[0]

    answer_start = int(inputs.shape[-1])
    pred = loaded_tokenizer.decode(outputs[answer_start:], skip_special_tokens=True)

    return pred

In [ ]:
def generate_cot_med42(row, question_type=None):
    """로드된 모델로 응답만 생성"""
    global loaded_model, loaded_tokenizer, device
    
    if loaded_model is None:
        raise ValueError("모델이 로드되지 않았습니다. 모델 로드 함수를 먼저 호출하세요.")
    
    system_base = cot_system_base
    MCQ_system = cot_system_base + cot_mcq_format
    MAQ_system = cot_system_base + cot_maq_format
    RQ_system = cot_system_base + cot_rq_format
    TFQ_system = cot_system_base + cot_tfq_format
    TFQ3_system = cot_system_base + cot_tfq3_format

    MCQ_template = cot_mcq_template_2shot
    MAQ_template = cot_maq_template_2shot
    RQ_template = cot_rq_template_2shot
    TFQ_template = cot_tfq_template_2shot    
    TFQ2_template = cot_tfq2_template_2shot
    TFQ3_template = cot_tfq3_template_2shot

    if question_type == "MCQ":
        system_prompt = MCQ_system
        user_template = MCQ_template
    elif question_type == "MAQ":
        system_prompt = MAQ_system
        user_template = MAQ_template
    elif question_type == "RQ":
        system_prompt = RQ_system
        user_template = RQ_template
    elif question_type == "TFQ":
        system_prompt = TFQ_system
        user_template = TFQ_template
    elif question_type == "TFQ2":
        system_prompt = TFQ_system
        user_template = TFQ2_template
    elif question_type == "TFQ3":
        system_prompt = TFQ3_system
        user_template = TFQ3_template

    # 프롬프트 생성
    user_input = user_template.format(ques=str(row['ques']))

    # Llama 3 형식의 대화 구조 생성
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]
    
    # Llama 3 전용 템플릿을 사용하여 프롬프트를 자동으로 생성
    # 이 방식은 모델이 기대하는 정확한 형식과 특수 토큰을 보장합니다.
    prompt = loaded_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 토크나이징 및 모델 추론
    inputs = loaded_tokenizer(prompt, return_tensors="pt").to(device)
    
    # 생성된 텍스트만 디코딩하기 위해 입력 토큰 길이를 저장
    input_length = inputs.input_ids.shape[1]
    
    outputs = loaded_model.generate(
        **inputs,
        max_new_tokens=2048, # max_length 대신 max_new_tokens 사용이 더 명확함
        eos_token_id=loaded_tokenizer.eos_token_id
    )
    
    # 새로 생성된 부분만 디코딩
    pred = loaded_tokenizer.decode(outputs[0, input_length:], skip_special_tokens=True)

    return pred

# AO, CoT 응답 생성

## medichat

In [ ]:
question_types = ["TFQ2"]
model_names = ["medichat"]
prompt_types = ["ao"]
data_path = "../generated_ques/"
output_pre_path = "output/TFQ23/regenerate/"


for model_name in model_names:
    print(f'[{model_name} 생성 시작]')
    for prompt_type in prompt_types:
        for q_type in question_types:
            print(f'{prompt_type}_{q_type} 생성 시작')
            process_questions(data_path = data_path, output_path=output_pre_path, prompt_type=prompt_type, model_name=model_name, question_type=q_type)

## med42

In [ ]:
question_types = ["TFQ2", "TFQ3"]
model_names = ["med42"]
prompt_types = ["cot", "ao"]
data_path = "../generated_ques/"
output_pre_path = "output/TFQ23/"


for model_name in model_names:
    print(f'[{model_name} 생성 시작]')
    for prompt_type in prompt_types:
        for q_type in question_types:
            print(f'{prompt_type}_{q_type} 생성 시작')
            process_questions(data_path = data_path, output_path=output_pre_path, prompt_type=prompt_type, model_name=model_name, question_type=q_type)

# CoVe

## 공통 함수

### 프롬프트
- select_prompt : Zero-shot
- select_prompt_full : 2-shot

In [ ]:
def select_prompt(prompt_type=None):

    ver_ques_system = '''
You need to extract important factual claims used as reasoning in the problem-solving process and create verification questions to check if these facts are accurate.

Your tasks are as follows:
1. Review the problem and its solution.
2. Identify the factual claims used by the student to derive the answer.
3. Create verification questions to check if these claims are accurate. (each question should be a single sentence)

You can make maximum 3 verification questions.
'''

    ver_ques_template = '''
Extract important factual claims from the context and create verification questions to check if these facts are accurate.
### Context :
{ques}
Answer: {cot}

### Created Verification Questions :
'''

    ver_ans_system = '''
You are a pharmaceutical expert. You should provide accurate and reliable answers to the given questions.

When answering, please follow these guidelines:
- **Write down each question**, and then answer it.
- If there are multiple questions, number each question with 1), 2), etc., and answer each one separately.
- Your answers should be clear and concise.
'''

    ver_ans_template = '''
Provide accurate answers to the given questions. You should write down each question and then answer each question separately.
Q: {ver_ques}
A:
'''

    final_ans_system = '''
You are studying pharmaceutical problems. After solving the problem, you searched for additional information about the parts you were unsure about.
Based on this new information, correct any errors in your initial solution (if any) and write your final response.
'''

    final_ans_template = '''
{ques}

Your initial response: {initial_response}

Let's check some facts.
{ver_ques_and_ans}

Using the information above, please solve the problem again.
Final Response : 
'''

    if prompt_type == '검증질문':
        return ver_ques_system, ver_ques_template
    elif prompt_type == '검증답변':
        return ver_ans_system, ver_ans_template
    elif prompt_type == '최종 답변':
        return final_ans_system, final_ans_template
    else:
        raise ValueError("유효하지 않은 프롬프트 유형입니다.")


In [ ]:
def select_full_prompt(prompt_type=None):

    ver_ques_system = '''
You need to extract important factual claims used as reasoning in the problem-solving process and create verification questions to check if these facts are accurate.

Your tasks are as follows:
1. Review the problem and its solution.
2. Identify the factual claims used by the student to derive the answer.
3. Create verification questions to check if these claims are accurate. (each question should be a single sentence)

When creating verification questions, consider the following:
- Make questions clear and specific.
- Write each question as a single sentence.
- Always follow this format when writing verification questions:
	<Fact to verify (from the given context)>, [Verification question to check this fact]
	<Fact to verify (from the given context)>, [Verification question to check this fact]
- for example:
    <Donepezil is a cholinesterase inhibitor used in the treatment of Alzheimer's disease.>, [Is donepezil a cholinesterase inhibitor used to treat Alzheimer's disease?]
    <Donepezil does not inhibit serotonin reuptake.>, [Does donepezil inhibit the reuptake of serotonin?]
- Make **maximum 3** verification questions.
'''

    ver_ques_template = '''
### Context 1:
Question: . Which of the following are not the mechanism of action of donepezil? There may be one or more correct choices.\nOptions: A: Activation of GABAergic transmission\nB: Irreversible blockade of dopamine receptors\nC: Selective and reversible inhibition of acetylcholinesterase enzyme\nD: Inhibition of serotonin reuptake”
Answer: Donepezil is a cholinesterase inhibitor used in the treatment of Alzheimer's disease. It works by selectively and reversibly inhibiting acetylcholinesterase, the enzyme responsible for breaking down acetylcholine in the synaptic cleft. Options A (Activation of GABAergic transmission), B (Irreversible blockade of dopamine receptors), and D (Inhibition of serotonin reuptake) are not mechanisms of action of donepezil. It does not affect GABA transmission, dopamine receptors, or serotonin reuptake.
Therefore, the answer is: A, B, D.

Extract important factual claims from the context and create maximum 3 verification questions to check if these facts are accurate.
### Response 1:
<Donepezil is a cholinesterase inhibitor used in the treatment of Alzheimer's disease.>, [Is donepezil a cholinesterase inhibitor used to treat Alzheimer's disease?]
<Donepezil does not inhibit serotonin reuptake.>, [Does donepezil inhibit the reuptake of serotonin?]

### Context 2:
Question: . Statement: \"Benzothiazepine best describes the classification of profenamine.\", is the statement above true or false? Please answer true/false.
Answer: Profenamine is a phenothiazine derivative used as an antiparkinsonian agent. Phenothiazines are a class of antipsychotic medications that share a common chemical structure characterized by a phenothiazine ring system. The statement claims that benzothiazepine best describes the classification of profenamine, which is false. Profenamine belongs to the phenothiazine class, not the benzothiazepine class. Benzothiazepines are a different class of drugs (like diltiazem) with different chemical structures and pharmacological properties.
Therefore, the answer is: false.

Extract important factual claims from the answer of the question above and create maximum 3 verification questions to check if these facts are accurate.
### Response 2:
<Profenamine is a phenothiazine derivative.>, [Is profenamine a derivative of phenothiazine?]
<Benzothiazepines are a different class of drugs than phenothiazines.>, [Are benzothiazepines in a different drug class than phenothiazines?]


### Context 3:
{ques}
Answer: {cot}

Extract important factual claims from the answer of the question above and create maximum 3 verification questions to check if these facts are accurate.
### Response 3:
'''

    ver_ans_system = '''
You are a pharmaceutical expert. You should provide accurate and reliable answers to the given questions.

When answering, please follow these guidelines:
- Write down each question, and then answer it.
- If there are multiple questions, number each question with 1), 2), etc., and answer each one separately.
    - Example: 1) <Question1>\n<Answer to Question1>\n 2)<Question2>\n<Answer to Question2>
- Your answers should be clear and concise.
'''

    ver_ans_template = '''
Q1: Is profenamine a derivative of phenothiazine?, Is donepezil a cholinesterase inhibitor used to treat Alzheimer's disease?
A1: 1) Is profenamine a derivative of phenothiazine?
Yes, profenamine is a phenothiazine derivative. It is also known as ethopropazine and is used as an antiparkinsonian agent. Its chemical structure is characterized by a phenothiazine ring system.
2) Is donepezil a cholinesterase inhibitor used to treat Alzheimer's disease?
Yes, donepezil is a cholinesterase inhibitor. It is a widely used medication specifically for the treatment of Alzheimer's disease.

Q2: Does moclobemide inhibit the reuptake of serotonin?
A2: Does moclobemide inhibit the reuptake of serotonin?
- No, moclobemide does not directly inhibit the reuptake of serotonin. It is a reversible inhibitor of monoamine oxidase-A (RIMA), which increases the levels of serotonin and other neurotransmitters by preventing their breakdown.

Q3: {ver_ques}
A3:
'''

    final_ans_system = '''
You are studying pharmaceutical problems. After solving the problem, you searched for additional information about the parts you were unsure about.
Based on this new information, correct any errors in your initial solution (if any) and write your final response.
The final answer must be written within the <final_answer>...</final_answer> tags.
'''

    final_ans_template = '''
### Example 1
Question: . Which of the following are not the mechanism of action of donepezil? There may be one or more correct choices.\nOptions: A: Activation of GABAergic transmission\nB: Irreversible blockade of dopamine receptors\nC: Selective and reversible inhibition of acetylcholinesterase enzyme\nD: Inhibition of serotonin reuptake

Initial Response: Donepezil's main function is to inhibit acetylcholinesterase, so C is a correct mechanism. The question asks for what is not a mechanism. Donepezil does not activate GABAergic transmission (A) or block dopamine receptors (B). While its primary action is on the cholinergic system, some sources suggest donepezil can have a minor effect on serotonin reuptake (D), especially at higher doses. Therefore, we will exclude D from the non-mechanisms. Therefore, the answer is: A, B.

Let's check some facts.
1) Does donepezil have a significant inhibitory effect on serotonin reuptake?
No, donepezil's inhibitory effect on serotonin reuptake is considered to be weak and not clinically significant. Its primary and therapeutic action is limited to the inhibition of acetylcholinesterase.
2) Is "Inhibition of serotonin reuptake" a widely accepted mechanism of action for donepezil?
No, "Inhibition of serotonin reuptake" is not a widely accepted or primary mechanism of action for donepezil in a pharmacological context. Its classification and clinical use are based on its role as a cholinesterase inhibitor.

Final Response(with <final_answer> tag) : Donepezil is a cholinesterase inhibitor used in the treatment of Alzheimer's disease. Its primary mechanism of action is the selective and reversible inhibition of acetylcholinesterase, as indicated in option C. The initial response incorrectly excluded D (Inhibition of serotonin reuptake) from the list of non-mechanisms based on a minor, non-clinically significant effect. However, new information from reliable sources confirms that this is not a primary or widely accepted mechanism of action for donepezil. Therefore, the options that are not mechanisms of action of donepezil are A (Activation of GABAergic transmission), B (Irreversible blockade of dopamine receptors), and D (Inhibition of serotonin reuptake). The corrected answer is: A, B, D.
<final_answer>A, B, D</final_answer>

### Example 2
Question: . Statement: \"Benzothiazepine best describes the classification of profenamine.\", is the statement above true or false? Please answer true/false.

Initial Response: Profenamine is a phenothiazine derivative used as an antiparkinsonian agent. Phenothiazines are a class of antipsychotic medications that share a common chemical structure characterized by a phenothiazine ring system. The statement claims that benzothiazepine best describes the classification of profenamine, which is false. Profenamine belongs to the phenothiazine class, not the benzothiazepine class. Benzothiazepines are a different class of drugs (like diltiazem) with different chemical structures and pharmacological properties. Therefore, the answer is: false.

Let's check some facts.
Are benzothiazepines in a different drug class than phenothiazines?
- Yes, benzothiazepines (e.g., diltiazem) are structurally and pharmacologically distinct from phenothiazines.

Final Response(with <final_answer> tag) : My initial response did not have any errors. Therefore, the answer remains false. <final_answer>false</final_answer>
------
Based on the format of the two examples above, please provide your answer.

{ques}

Initial Response: {initial_response}

Let's check some facts.
{ver_ques_and_ans}

Final Response(with <final_answer> tag) : 
'''

    if prompt_type == '검증질문':
        return ver_ques_system, ver_ques_template
    elif prompt_type == '검증답변':
        return ver_ans_system, ver_ans_template
    elif prompt_type == '최종 답변':
        return final_ans_system, final_ans_template
    else:
        raise ValueError("유효하지 않은 프롬프트 유형입니다.")


### 검증질문 파싱 함수
- ver_ques_raw에서 [] 안의 검증 질문을 파싱하는 함수

In [ ]:
def parse_verification_questions(ver_ques_raw):

    system = '''당신은 주어진 텍스트에서 검증 질문(Verification Question)들만을 파싱한 후, 쉼표로 연결하여 반환하는 역할을 합니다.
검증 질문들을 쉼표로 연결한 결과만 출력하고, 이 이외의 설명이나 문장은 출력하지 않습니다.

예를 들어, 다음과 같은 텍스트가 주어지면:
Verification question: Is donepezil a cholinesterase inhibitor used to treat Alzheimer's disease?
Verification question: Is serotonin antagonists involved in preventing migraine attacks at low doses?

당신의 출력은 다음과 같아야 합니다:
Is donepezil a cholinesterase inhibitor used to treat Alzheimer's disease?, Is serotonin antagonists involved in preventing migraine attacks at low doses?'''
    template = ''' 다음 텍스트에서 검증 질문들만을 파싱하여 쉼표로 연결한 결과를 반환하세요.

    {ver_ques_raw}'''
    custom_llm = ChatOpenAI(
        openai_api_key=OPENAI_API_KEY,
        model_name="gpt-4.1",
        max_tokens=5000
    )
    custom_prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("user", template)
    ])
    chain = custom_prompt | custom_llm
            
    # 입력 데이터 준비
    input_data = {
        'ver_ques_raw': str(ver_ques_raw)
    }
            
    # 체인 실행
    response = chain.invoke(input_data)
    return response.content.strip()
    

### 검증질문, 검증답변, 최종답변 생성 함수

In [ ]:
# 검증 질문 - 메인 실행 코드
def process_verification_questions(data_path, output_path, model_name, question_type):
    """검증질문을 생성하고 엑셀 파일로 저장하는 메인 함수"""
    
    # 데이터 로드
    df = pd.read_excel(data_path + f'{model_name}_{question_type}_cot.xlsx')
    print(f"데이터 로드 완료: {len(df)}개 행")
    
    # ver_ques_raw 컬럼 초기화
    df['ver_ques_raw'] = ''
    df['ver_ques'] = ''
    
    # 각 행에 대해 검증질문 생성 (tqdm으로 진행상황 표시)
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="검증질문 생성 중"):
        try:
            # 검증질문 생성
            ver_ques_raw = generate_response(row, prompt_type='검증질문', max_length=4096)
            df.at[idx, 'ver_ques_raw'] = ver_ques_raw
            
            # 검증질문 파싱
            ver_ques = parse_verification_questions(ver_ques_raw)
            df.at[idx, 'ver_ques'] = ver_ques
            
        except Exception as e:
            print(f"\n오류 발생 (행 {idx+1}): {str(e)}")
            df.at[idx, 'ver_ques_raw'] = f"오류: {str(e)}"
            df.at[idx, 'ver_ques'] = ""
    
    # 결과 저장
    output_pre_path = output_path
    if not os.path.exists(output_pre_path):
        os.makedirs(output_pre_path)  
    output_path = output_pre_path + f'{model_name}_{question_type}_cove1_ver_ques.xlsx'
    df.to_excel(output_path, index=False)
    print(f"\n결과 저장 완료: {output_path}")
    
    return df

In [ ]:
# 검증 답변 - 메인 실행 코드
def process_verification_answers(output_path, model_name, question_type):
    """검증답변을 생성하고 엑셀 파일로 저장하는 메인 함수"""
    
    # 데이터 로드
    df = pd.read_excel(output_path + f'{model_name}_{question_type}_cove1_ver_ques.xlsx')
    print(f"데이터 로드 완료: {len(df)}개 행")
    
    # ver_q_and_a 컬럼 초기화
    df['ver_q_and_a'] = ''
    
    # 각 행에 대해 검증답변 생성 (tqdm으로 진행상황 표시)
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="검증답변 생성 중"):
        try:
            # 검증질문 생성
            ver_q_and_a = generate_response(row, prompt_type='검증답변', max_length=4096)
            df.at[idx, 'ver_q_and_a'] = ver_q_and_a
            
        except Exception as e:
            print(f"\n오류 발생 (행 {idx+1}): {str(e)}")
            df.at[idx, 'ver_q_and_a'] = f"오류: {str(e)}"
    
    # 결과 저장
    output_pre_path = output_path
    output_path = output_pre_path + f'{model_name}_{question_type}_cove2_ver_ans.xlsx'
    df.to_excel(output_path, index=False)
    print(f"\n결과 저장 완료: {output_path}")
    
    return df

In [ ]:
# 최종 답변 - 메인 실행 코드

def process_final_answer(output_path, model_name, question_type):
    """수정해 최종 답변을 생성하고 엑셀 파일로 저장하는 메인 함수"""
    
    output_pre_path = output_path
    temp_path = output_pre_path + f"{model_name}_{question_type}_cove3_final_ans_temp.xlsx"
    output_path = output_pre_path + f'{model_name}_{question_type}_cove3_final_ans.xlsx'

    # 데이터 로드
    df = pd.read_excel(output_pre_path + f'{model_name}_{question_type}_cove2_ver_ans.xlsx')
    print(f"데이터 로드 완료: {len(df)}개 행")
    
    # ver_q_and_a 컬럼 초기화
    df['final_response'] = ''
    df['final_ans'] = ''
    
    # 각 행에 대해 검증답변 생성 (tqdm으로 진행상황 표시)
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="최종답변 생성 중"):
        try:
            # 검증질문 생성
            final_response = generate_response(row, prompt_type='최종 답변', max_length=7000)
            df.at[idx, 'final_response'] = final_response
            
            # row에 final_response 추가
            row['final_response'] = final_response 
            
            # 최종 답변 파싱
            gpt_response = parse_final_answer(row, question_type, prompt_type="cove") #gpt로 파싱결과 생성

            if question_type == 'RQ': # RQ일 경우 태그 형식으로 출력된 결과 파싱
                df.at[idx, 'parse_raw'] = gpt_response
                match = re.search(r'<final_answer>(.*?)</final_answer>', gpt_response, re.DOTALL)
                if match:
                    final_ans = match.group(1).strip()
                else:
                    # 태그가 없으면 GPT 응답 그대로 사용
                    final_ans = gpt_response
            else: # 나머지 경우 그대로 사용
                final_ans = gpt_response

            df.at[idx, 'final_ans'] = final_ans
            
            # 임시 저장 (진행상황 보존)
            df.to_excel(temp_path, index=False)
            
        except Exception as e:
            print(f"\n오류 발생 (행 {idx+1}): {str(e)}")
            df.at[idx, 'final_response'] = f"오류: {str(e)}"
            df.at[idx, 'final_ans'] = ""

    
    # 결과 저장
    df.to_excel(output_path, index=False)
    print(f"\n결과 저장 완료: {output_path}")

    # 임시저장 파일 제거
    if os.path.exists(temp_path):
        os.remove(temp_path)
        print("임시저장 파일 제거 완료")
    
    return df

## medicine-chat-7B

### 함수 정의

In [ ]:
# 응답 생성 함수 (llama2 계열 모델)

def generate_response(row, prompt_type=None, max_length=4096):
    """로드된 모델로 응답만 생성"""
    global loaded_model, loaded_tokenizer, device
    
    # 프롬프트 로드
    if loaded_model is None:
        raise ValueError("모델이 로드되지 않았습니다. 모델 로드 함수를 먼저 호출하세요.")
    
    system_prompt = select_prompt(prompt_type)[0]
    user_template = select_prompt(prompt_type)[1]

    # 프롬프트 생성
    if prompt_type == '검증질문':
        user_input = user_template.format(ques=str(row['ques']), cot=str(row['cot']))
    elif prompt_type == '검증답변':
        user_input = user_template.format(ver_ques=str(row['ver_ques']))
    elif prompt_type == '최종 답변':
        user_input = user_template.format(ques=str(row['ques']), initial_response=str(row['cot']), ver_ques_and_ans=str(row['ver_q_and_a']))
    
    prompt = f"<s>[INST] <<SYS>>{system_prompt}<</SYS>>\n\n{user_input} [/INST]"
    
    # 토크나이징 및 모델 추론
    inputs = loaded_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    outputs = loaded_model.generate(input_ids=inputs, max_length=max_length)[0]

    answer_start = int(inputs.shape[-1])
    pred = loaded_tokenizer.decode(outputs[answer_start:], skip_special_tokens=True)

    return pred

### 파이프라인

In [ ]:
input_path = 'output/TFQ3/' # cot 파일이 있는 경로
output_path = 'output/TFQ3/CoVe/'
model_names = ['medichat']
question_types = ['TFQ3']

for model in model_names:
    for question_type in question_types:

        print('='*50)
        print(f"Processing {model}_{question_type}...")
        
        # 1. 검증 질문 생성
        ver_ques_file = output_path + f'{model}_{question_type}_cove1_ver_ques.xlsx'
        if os.path.exists(ver_ques_file):
            print(f"  - 검증 질문 파일이 이미 존재합니다: {ver_ques_file}")
            ver_ques_df = pd.read_excel(ver_ques_file)
        else:
            print(f"  - 검증 질문 생성 중...")
            ver_ques_df = process_verification_questions(input_path, output_path, model, question_type)
        
        # 2. 검증 답변 생성
        ver_ans_file = output_path + f'{model}_{question_type}_cove2_ver_ans.xlsx'
        if os.path.exists(ver_ans_file):
            print(f"  - 검증 답변 파일이 이미 존재합니다: {ver_ans_file}")
            ver_ans_df = pd.read_excel(ver_ans_file)
        else:
            print(f"  - 검증 답변 생성 중...")
            ver_ans_df = process_verification_answers(output_path, model, question_type)
        
        # 3. 최종 답변 생성
        final_ans_file = output_path + f'{model}_{question_type}_cove3_final_ans.xlsx'
        if os.path.exists(final_ans_file):
            print(f"  - 최종 답변 파일이 이미 존재합니다: {final_ans_file}")
            final_ans_df = pd.read_excel(final_ans_file)
        else:
            print(f"  - 최종 답변 생성 중...")
            final_ans_df = process_final_answer(output_path, model, question_type)
        
        print(f"Completed {model}_{question_type}!")

## Med42-70B

### 함수 정의

응답 생성 함수

In [ ]:
# 응답 생성 함수 (llama3 계열 모델델)

def generate_response(row, prompt_type=None, max_length=4096):
    """로드된 모델로 응답만 생성"""
    global loaded_model, loaded_tokenizer, device
    
    # 프롬프트 로드
    if loaded_model is None:
        raise ValueError("모델이 로드되지 않았습니다. 모델 로드 함수를 먼저 호출하세요.")
    
    system_prompt = select_full_prompt(prompt_type)[0]
    user_template = select_full_prompt(prompt_type)[1]

    # 프롬프트 생성
    if prompt_type == '검증질문':
        user_input = user_template.format(ques=str(row['ques']), cot=str(row['cot']))
    elif prompt_type == '검증답변':
        user_input = user_template.format(ver_ques=str(row['ver_ques']))
    elif prompt_type == '최종 답변':
        user_input = user_template.format(ques=str(row['ques']), initial_response=str(row['cot']), ver_ques_and_ans=str(row['ver_q_and_a']))
    
    # Llama 3 형식의 대화 구조 생성
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]
    
    # Llama 3 전용 템플릿을 사용하여 프롬프트를 자동으로 생성
    prompt = loaded_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 토크나이징 및 모델 추론
    inputs = loaded_tokenizer(prompt, return_tensors="pt").to(device)
    
    # 생성된 텍스트만 디코딩하기 위해 입력 토큰 길이를 저장
    input_length = inputs.input_ids.shape[1]
    
    outputs = loaded_model.generate(
        **inputs,
        max_new_tokens=max_length, # max_length 대신 max_new_tokens 사용이 더 명확함
        eos_token_id=loaded_tokenizer.eos_token_id
    )
    
    # 새로 생성된 부분만 디코딩
    pred = loaded_tokenizer.decode(outputs[0, input_length:], skip_special_tokens=True)

    return pred


### 파이프라인

In [ ]:
input_path = 'output/TFQ23/' # cot 파일이 있는 경로
output_path = 'output/TFQ23/CoVe/'
model_names = ['med42']
question_types = ['TFQ2','TFQ3']

for model in model_names:
    for question_type in question_types:

        print('='*50)
        print(f"Processing {model}_{question_type}...")
        
        # 1. 검증 질문 생성
        ver_ques_file = output_path + f'{model}_{question_type}_cove1_ver_ques.xlsx'
        if os.path.exists(ver_ques_file):
            print(f"  - 검증 질문 파일이 이미 존재합니다: {ver_ques_file}")
            ver_ques_df = pd.read_excel(ver_ques_file)
        else:
            print(f"  - 검증 질문 생성 중...")
            ver_ques_df = process_verification_questions(input_path, output_path, model, question_type)
        
        # 2. 검증 답변 생성
        ver_ans_file = output_path + f'{model}_{question_type}_cove2_ver_ans.xlsx'
        if os.path.exists(ver_ans_file):
            print(f"  - 검증 답변 파일이 이미 존재합니다: {ver_ans_file}")
            ver_ans_df = pd.read_excel(ver_ans_file)
        else:
            print(f"  - 검증 답변 생성 중...")
            ver_ans_df = process_verification_answers(output_path, model, question_type)
        
        # 3. 최종 답변 생성
        final_ans_file = output_path + f'{model}_{question_type}_cove3_final_ans.xlsx'
        if os.path.exists(final_ans_file):
            print(f"  - 최종 답변 파일이 이미 존재합니다: {final_ans_file}")
            final_ans_df = pd.read_excel(final_ans_file)
        else:
            print(f"  - 최종 답변 생성 중...")
            final_ans_df = process_final_answer(output_path, model, question_type)
        
        print(f"Completed {model}_{question_type}!")

# 정확도 계산

In [ ]:
import pandas as pd
data_path = 'output/'
model_names = ['med42']
question_types = ['MCQ', 'MAQ', 'TFQ','RQ']

print("[1. Answer-Only 정확도]")
for model_name in model_names:
    print(f"모델: {model_name}")
    for question_type in question_types:
        df = pd.read_excel(data_path + f'{model_name}_{question_type}_ao.xlsx')
        ao_corr_count = len(df[df['ans']==df['label']])
        ao_total_count = len(df)
        ao_accuracy = ao_corr_count / ao_total_count

        print(f'{model}_{question_type} AO 정확도: {initial_accuracy:.2%}')

print("\n==========================================\n")

print("[2. CoT 정확도]")
for model_name in model_names:
    print(f"모델: {model_name}")
    for question_type in question_types:
        df = pd.read_excel(data_path + f'{model_name}_{question_type}_cot.xlsx')
        initial_corr_count = len(df[df['ans']==df['label']])
        initial_total_count = len(df)
        initial_accuracy = initial_corr_count / initial_total_count

        print(f'{model}_{question_type} CoT 정확도: {initial_accuracy:.2%}')

print("\n==========================================\n")

print("[3. CoVe 정확도]")
for model_name in model_names:
    print(f"모델: {model_name}")
    for question_type in question_types:
        df = pd.read_excel(data_path + f'{model_name}_{question_type}_cove3_final_ans.xlsx')
        final_corr_count = len(df[df['final_ans']==df['label']])
        final_total_count = len(df)
        final_accuracy = final_corr_count / final_total_count

        print(f'{model}_{question_type} CoVe 정확도: {final_accuracy:.2%}')